# IJCAI 2027 — Luồng G2 trên Colab GPU: GCN `TTC-Cone` (5 Seeds, ETH/UCY + SDD) + `MOT-Glitch` + `TTC-OSC`

Notebook này chuyển toàn bộ khối lượng của kernel Kaggle thứ 2 (`ijcai-osc-mot-g2` bị `QUEUED`) sang chạy thẳng trên **Google Colab GPU**, lưu tiến độ trực tiếp vào **Google Drive (`MyDrive/TTC_STGCNN_IJCAI2027/g2_work`)**:
1. Huấn luyện **5 seeds × 5 tập ETH/UCY (`150` mô hình)** cho bộ đồ thị GCN & Nón Nhân quả Động học (`B`, `K`, `K0`, `KR`, `KN`, **`KC = A_cone`**) + đối chứng `TFW` & **`TFC1`**. 
2. Mở rộng sang **Stanford Drone Dataset (`SDD`, `17` mô hình)** đóng gói sẵn gọn nhẹ trong repo (không cần tải 5 GB video).
3. Tự động chạy hậu kiểm toàn diện `src/ijcai_probe.py`: đường cong độ nhạy theo khoảng cách, `out_cone_max`, sai số cặp lao nhanh `Appr. ADE`, kiểm thử nhiễu bộ bám vết **`MOT-Glitch`** (`ID-switch`, `FOV drop`, `Ghost FP`), và **`TTC-OSC` (1/2/3-sweep + dải `JADE`)**.

### Hướng dẫn chạy:
1. Chọn **Runtime → Change runtime type → GPU**.
2. Bấm **Runtime → Run all** (hỗ trợ chạy tiếp tự động nếu rớt mạng).
3. Khi xong Ô 4, tải file gọn nhẹ `g2_osc_mot_summary.zip` trong `MyDrive/TTC_STGCNN_IJCAI2027/` về máy.

In [ ]:
# Ô 1: Mount Google Drive, kéo code + gói SDD gọn nhẹ từ GitHub (idea1-colab) & chạy Self-Test
import os, sys, subprocess, shutil, glob, json
from google.colab import drive

drive.mount('/content/drive')

REPO_URL = 'https://github.com/Thanh-2312/TTC-STGCNN.git'
BRANCH = 'idea1-colab'
DRIVE_BASE = '/content/drive/MyDrive/TTC_STGCNN_IJCAI2027'
REPO_DIR = os.path.join(DRIVE_BASE, 'repo_g2')
WORK_DIR = os.path.join(DRIVE_BASE, 'g2_work')
os.makedirs(DRIVE_BASE, exist_ok=True)

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', '-b', BRANCH, '--single-branch', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', f'origin/{BRANCH}'], check=True)

RUNNER = os.path.join(REPO_DIR, 'colab', 'g2_colab', 'run_g2_colab.py')
PROBE = os.path.join(REPO_DIR, 'src', 'ijcai_probe.py')
assert os.path.exists(RUNNER), f'Không tìm thấy {RUNNER}'

import torch
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU', '| CPU cores:', os.cpu_count())
assert torch.cuda.is_available(), 'Chưa bật GPU! Hãy chọn Runtime -> Change runtime type -> GPU.'

subprocess.run([sys.executable, PROBE, '--selftest'], check=True)

def run_g2(**kw):
    env = dict(os.environ, **{k: str(v) for k, v in kw.items()})
    p = subprocess.Popen([sys.executable, '-u', RUNNER], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
    return p.wait()

In [ ]:
# Ô 2: Smoke Test nhanh (~1-2 phút trên cả ETH/UCY zara1 và SDD, 1 epoch) để xác nhận toàn bộ pipeline G2
SMOKE_DIR = '/content/g2_smoke'
shutil.rmtree(SMOKE_DIR, ignore_errors=True)
rc = run_g2(G2_WORK=SMOKE_DIR, G2_TMP='/content/g2_tmp_smoke', G2_SMOKE='1', G2_WORKERS=4)

failed_s = json.load(open(os.path.join(SMOKE_DIR, 'failed.json'))) if os.path.exists(os.path.join(SMOKE_DIR, 'failed.json')) else ['missing']
print('\n=== KẾT QUẢ SMOKE TEST G2 ===')
print('Số job lỗi:', len(failed_s), failed_s)
sum_md = os.path.join(SMOKE_DIR, 'out', 'ijcai_k2_summary.md')
if os.path.exists(sum_md):
    print('\n' + open(sum_md, encoding='utf-8').read())

In [ ]:
# Ô 3: Chạy ĐẦY ĐỦ 177 mô hình (ETH/UCY + SDD) + Hậu kiểm MOT-Glitch & TTC-OSC — lưu thẳng vào Google Drive
workers = 6
print(f'Bắt đầu chạy đầy đủ G2 vào {WORK_DIR} với {workers} tiến trình GPU song song...')
run_g2(G2_WORK=WORK_DIR, G2_TMP='/content/g2_tmp_full', G2_WORKERS=workers)

sum_md = os.path.join(WORK_DIR, 'out', 'ijcai_k2_summary.md')
if os.path.exists(sum_md):
    print('\n' + '='*80 + '\n')
    print(open(sum_md, encoding='utf-8').read())

In [ ]:
# Ô 4: Đóng gói kết quả gọn nhẹ trên Google Drive để tải về máy cho phiên phân tích tiếp theo
import zipfile

summary_zip = os.path.join(DRIVE_BASE, 'g2_osc_mot_summary.zip')
out_root = os.path.join(WORK_DIR, 'out')
with zipfile.ZipFile(summary_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for fn in ('ijcai_k2_summary.json', 'ijcai_k2_summary.md', 'manifest.json'):
        fp = os.path.join(out_root, fn)
        if os.path.exists(fp):
            z.write(fp, fn)
    fp_fail = os.path.join(WORK_DIR, 'failed.json')
    if os.path.exists(fp_fail):
        z.write(fp_fail, 'failed.json')
    for rj in sorted(glob.glob(os.path.join(out_root, '*_IJCAI-*', 'result.json'))):
        rel = os.path.relpath(rj, out_root).replace(os.sep, '/')
        z.write(rj, 'runs/' + rel)

done_runs = len(glob.glob(os.path.join(out_root, '*_IJCAI-*', 'result.json')))
print(f'Đã hoàn thành {done_runs} mô hình.')
print('File kết quả gọn nhẹ (tải file này về máy):', summary_zip, f'({os.path.getsize(summary_zip)/1024:.1f} KB)')